# Stany magazynowe

Krok ETL po `transform`. Dodaje do tabeli dwie kolumny stanu magazynowego:
- **`StanPoRuchu`** — stan towaru po każdym wierszu, narastająco w kolejności `Data` → `KolejnyWDniu` → `Kolejnosc`,
- **`StanKoniecDnia`** — ostatni `StanPoRuchu` dnia, powtórzony we wszystkich wierszach SKU × dzień (w analizach **nie sumować**, brać `first()` / `max()`).

**Metoda:**
- ruch = `RuchMagazynowy` (z `transform`, ze znakiem); dokumenty nieaktywne (`AktywnyDok = 0`) nie zmieniają stanu,
- REM i BO (`MetodaLiczenia = 'RESET_IP'`) **resetują stan** do ilości ze spisu — w wierszu resetu `RuchMagazynowy` = `IloscPlus` (stan ze spisu, nie zmiana); pozycje tego samego towaru w jednym dokumencie się sumują,
- puste pozycje spisu (`IloscPlus = 0` i `IloscMinus = 0`) nie resetują,
- wiersze zerowe z `calendar` (`DokId = -1`, `KolejnyWDniu = -1`) mają ruch 0 — przenoszą stan z poprzedniego dnia,
- przed pierwszym resetem towaru stan liczony od 0.

**Kontrola jakości:** stan wyliczony tuż przed każdym REM vs `IloscMinus` (stan systemowy przed spisem).

Wejście: `dane/interim/fact_inka_final.parquet`. Wyjście: `dane/interim/fact_inka_final_stany.parquet` (wejście + 2 kolumny, ta sama liczba wierszy). Szczegóły i wyniki: `docs/stany_hiu.md`.

In [1]:
# Importy i ustawienie katalogu głównego projektu
import sys, os
import pandas as pd
import numpy as np

# Przejdź do katalogu głównego projektu tylko raz — bezpieczne przy wielokrotnym uruchomieniu
if os.path.basename(os.getcwd()) != "analiza_sprzedazy_w_sklepie":
    os.chdir("../..")

sys.path.insert(0, os.getcwd())
import utils.tools as tools

## [PARAMETRY]

In [2]:
WEJSCIE      = "dane/interim/fact_inka_final.parquet"
WYJSCIE      = "dane/interim/fact_inka_final_stany.parquet"
KOLUMNY_HASH = ['SKU', 'Data', 'DokId', 'Kolejnosc']    # sortowanie do sumy kontrolnej (jak w transform)
KOLEJNOSC    = ['SKU', 'Data', 'KolejnyWDniu', 'Kolejnosc']   # kolejność ruchów przy liczeniu stanu
ZAOKRAGLENIE = 4        # miejsca po przecinku — sumy ułamków (towary ważone) dają np. 2.34499999
TOLERANCJA   = 0.001    # różnica uznawana za zgodność w kontroli REM
TYPDOK_REM   = 16       # remanent (BO = 14 też resetuje, ale nie ma stanu systemowego do porównania)

## [FUNKCJE]

In [3]:
def ruch_do_stanu(df):
    """Zmiana stanu w wierszu: RuchMagazynowy dla aktywnych dokumentów, 0 dla pozostałych.

    W wierszach resetu (RESET_IP) wartość = ilość ze spisu — sumowana od zera w nowej epoce.
    """
    return df['RuchMagazynowy'].where(df['AktywnyDok'] == 1, 0.0)


def czy_reset(df):
    """Wiersze resetujące stan: niepusta pozycja REM/BO z aktywnego dokumentu."""
    pusta = (df['IloscPlus'] == 0) & (df['IloscMinus'] == 0)
    return (df['MetodaLiczenia'] == 'RESET_IP') & (df['AktywnyDok'] == 1) & ~pusta


def poczatek_resetu(df, reset):
    """Pierwsza pozycja resetu w dokumencie (SKU × DokId) — od niej zaczyna się nowa epoka.

    Kolejne pozycje tego samego dokumentu dodają się do stanu (spis sumowany po pozycjach).
    Wymaga posortowanego df.
    """
    pierwsza_resetu = ~df[reset].duplicated(['SKU', 'DokId'])
    return reset & pierwsza_resetu.reindex(df.index, fill_value=False)


def stan_po_ruchu(df, ruch, start):
    """Stan po każdym wierszu: suma narastająca ruchów w epoce (od resetu do resetu) per SKU.

    Przed pierwszym resetem stan liczony od 0. Wymaga posortowanego df.
    """
    epoka = start.groupby(df['SKU']).cumsum()
    return ruch.groupby([df['SKU'], epoka]).cumsum().round(ZAOKRAGLENIE)


def stan_koniec_dnia(df):
    """Ostatni StanPoRuchu dnia, powtórzony we wszystkich wierszach SKU × Data. Wymaga posortowanego df."""
    return df.groupby(['SKU', 'Data'])['StanPoRuchu'].transform('last')


def kontrola_rem(df, reset, start):
    """Stan wyliczony tuż przed każdym REM vs stan systemowy przed spisem (suma IloscMinus dokumentu).

    Zwraca jeden wiersz na SKU × dokument REM z kolumnami StanWyliczony, StanSystemowy, Roznica, Zgodny.
    """
    stan_przed = df.groupby('SKU')['StanPoRuchu'].shift(1).fillna(0.0)   # stan po poprzednim wierszu
    rem = df[reset & (df['TypDok'] == TYPDOK_REM)]
    wynik = rem.groupby(['SKU', 'DokId'], as_index=False).agg(
        Data=('Data', 'first'), CzyWazony=('CzyWazony', 'first'), StanSystemowy=('IloscMinus', 'sum'))
    pierwsza_rem = start & (df['TypDok'] == TYPDOK_REM)
    przed = df.loc[pierwsza_rem, ['SKU', 'DokId']].assign(StanWyliczony=stan_przed[pierwsza_rem])
    wynik = wynik.merge(przed, on=['SKU', 'DokId'], how='left')
    wynik['Roznica'] = (wynik['StanWyliczony'] - wynik['StanSystemowy']).round(ZAOKRAGLENIE)
    wynik['Zgodny'] = wynik['Roznica'].abs() <= TOLERANCJA
    return wynik

## Wczytanie

In [4]:
# Suma kontrolna wejścia — ten sam hash u obu osób = identyczne dane
moj_hash = tools.hash_danych_bezpieczny(WEJSCIE, kolumny_sortowania=KOLUMNY_HASH)
print(f"Mój hash (posortowane):   {os.path.basename(WEJSCIE)}   {moj_hash}")

Mój hash (posortowane):   fact_inka_final.parquet   b73691d7bc6fef407c5e40bb256ad261b4c1c28537d0a19b98ef5ec89bf62bef


Hash `fact_inka_final.parquet` (z wyjścia `transform.ipynb`):  
**b73691d7bc6fef407c5e40bb256ad261b4c1c28537d0a19b98ef5ec89bf62bef**

In [5]:
df = pd.read_parquet(WEJSCIE)
n_wejscie = len(df)
print(f"Wierszy: {n_wejscie:,}, kolumn: {df.shape[1]}")

Wierszy: 9,485,872, kolumn: 60


## Obliczenia

Sortowanie `SKU` → `Data` → `KolejnyWDniu` → `Kolejnosc` (wiersz zerowy dnia ma `KolejnyWDniu = -1`, więc stoi na początku dnia).

In [6]:
df = df.sort_values(KOLEJNOSC, kind='stable').reset_index(drop=True)

ruch  = ruch_do_stanu(df)
reset = czy_reset(df)
start = poczatek_resetu(df, reset)

df['StanPoRuchu']    = stan_po_ruchu(df, ruch, start)
df['StanKoniecDnia'] = stan_koniec_dnia(df)

print(f"Pozycji resetu: {reset.sum():,}, dokumentów resetu (SKU × DokId): {start.sum():,}")

Pozycji resetu: 24,175, dokumentów resetu (SKU × DokId): 24,174


## [WALIDACJA]

In [7]:
# Kontrole techniczne
assert len(df) == n_wejscie, "Liczba wierszy różna od wejścia"
assert df['StanPoRuchu'].notna().all() and df['StanKoniecDnia'].notna().all(), "NaN w stanach"
assert (df.groupby(['SKU', 'Data'])['StanKoniecDnia'].nunique() == 1).all(), "StanKoniecDnia niestały w dniu"

# w pierwszej pozycji resetu stan = ilość ze spisu
assert np.allclose(df.loc[start, 'StanPoRuchu'], df.loc[start, 'RuchMagazynowy'].round(ZAOKRAGLENIE)), "Reset nie ustawia stanu"

# StanKoniecDnia = StanPoRuchu w ostatnim wierszu dnia
ostatni_w_dniu = ~df.duplicated(['SKU', 'Data'], keep='last')
assert (df.loc[ostatni_w_dniu, 'StanKoniecDnia'] == df.loc[ostatni_w_dniu, 'StanPoRuchu']).all(), "StanKoniecDnia ≠ ostatni StanPoRuchu"
print("Kontrole techniczne OK")

Kontrole techniczne OK


In [8]:
# Kontrola jakości: stan wyliczony przed REM vs stan systemowy (IloscMinus)
kontrola = kontrola_rem(df, reset, start)
print(f"REM (SKU × dokument): {len(kontrola):,}, zgodnych: {kontrola['Zgodny'].mean():.1%}")
print(kontrola.groupby('CzyWazony')['Zgodny'].agg(['size', 'mean']).rename(index={0: 'nieważone', 1: 'ważone'}))

REM (SKU × dokument): 18,756, zgodnych: 95.1%
            size      mean
CzyWazony                 
nieważone  18066  0.963578
ważone       690  0.610145


In [9]:
# Przegląd: ujemne stany — głównie artefakty danych (dokumenty wpisane później z wcześniejszą datą,
# PZ zbiorczy, start od 0 przed pierwszym REM); szczegóły w docs/stany_hiu.md
dzien = df.drop_duplicates(['SKU', 'Data'])
print(f"SKU-dni z ujemnym StanKoniecDnia: {(dzien['StanKoniecDnia'] < 0).mean():.2%}")
print(f"SKU z ujemnym stanem choć raz: {dzien.loc[dzien['StanKoniecDnia'] < 0, 'SKU'].nunique():,} z {dzien['SKU'].nunique():,}")

SKU-dni z ujemnym StanKoniecDnia: 1.58%
SKU z ujemnym stanem choć raz: 2,821 z 12,481


## Zapis + hash

In [10]:
df.to_parquet(WYJSCIE, compression='zstd', index=False)
print(f"Zapisano: {df.shape}")

Zapisano: (9485872, 62)


In [11]:
# Suma kontrolna wyniku — przelicz po każdej zmianie kodu
moj_hash = tools.hash_danych_bezpieczny(WYJSCIE, kolumny_sortowania=KOLUMNY_HASH)
print(f"Mój hash (posortowane):   {os.path.basename(WYJSCIE)}   {moj_hash}")

Mój hash (posortowane):   fact_inka_final_stany.parquet   47e18bec02828a9a98231c152378a45465d69171e71e6017e4161ca9e237a6b1


Hash końcowy `fact_inka_final_stany.parquet`:  
**47e18bec02828a9a98231c152378a45465d69171e71e6017e4161ca9e237a6b1**